In [1]:
import os
import requests

In [24]:
language_to_writing_system = {
        "marathi": ["devanagari"], 
        "hindi": ["devanagari"], 
        "sanskrit": ["devanagari"],
        "tamil": ["tamil"], 
        "telugu": ["telugu"], 
        "kannada": ["kannada"],
        "malayalam": ["malayalam"],
        "bengali": ["bengali"], 
        "assamese": ["bengali"],
        "manipuri": ["bengali", "meetei-mayek"],
        "gujarati": ["gujarati"], 
        "punjabi": ["gurmukhi"], 
        "oriya": ["odia"],
        "kashmiri": ["devanagari", "arabic"], 
        "sindhi": ["arabic", "devanagari"], 
        "urdu": ["arabic"],
        "english": ["latin"], 
        "santali": ["ol-chiki", "devanagari"],
    }

In [27]:
writing_system_to_language = {
        'devanagari': ['marathi','hindi','sanskrit','kashmiri','sindhi','santali'],
        'tamil': ['tamil'],
        'telugu': ['telugu'],
        'kannada': ['kannada'],
        'malayalam': ['malayalam'],
        'bengali': ['bengali', 'assamese', 'manipuri'],
        'meetei-mayek': ['manipuri'],
        'gujarati': ['gujarati'],
        'gurmukhi': ['punjabi'],
        'odia': ['oriya'],
        'arabic': ['kashmiri', 'sindhi', 'urdu'],
        'latin': ['english'],
        'ol-chiki': ['santali']
 }

In [21]:
def get_specific_font_styles(writing_system, download_dir=".", styles_to_download=None):
    """
    Finds and downloads specific styles (regular, bold, italic) for the top 5 most popular Google Fonts for a given writing system.

    Args:
        language (str): Writing System (e.g., 'devanagari', 'latin'). Many Indic languages could have the same writing system.
        download_dir (str, optional): The main directory to save
        styles_to_download (list, optional): A list of styles to download.
            Defaults to ["regular", "bold", "italic", "bold-italic"].
    """
    # Define the default styles and their mapping to Google Fonts API variant names
    if styles_to_download is None:
        styles_to_download = ["regular", "bold", "italic", "bold-italic"]

    style_to_variant_map = {
        "regular": "regular",
        "italic": "italic",
        "bold": "700",
        "bold-italic": "700italic"
    }

    api_key = os.getenv("GOOGLE_FONTS_API")

    # Determine the actual variant names to look for
    target_variants = [style_to_variant_map[style] for style in styles_to_download if style in style_to_variant_map]
    top_popularity = 5

    print(f"Targeting styles: {styles_to_download} (API variants: {target_variants})\n")

    api_url = f"https://www.googleapis.com/webfonts/v1/webfonts?key={api_key}&subset={writing_system}&sort=popularity&sort=style"
    print(f"--- Querying for script: {writing_system.capitalize()} ---")
    
    try:
        response = requests.get(api_url)
        response.raise_for_status()
        fonts_data = response.json()
        if not fonts_data.get("items"):
            print(f"No fonts found for the '{writing_system}' ")

        script_folder = os.path.join(download_dir, writing_system.capitalize())
        if not os.path.exists(script_folder): os.makedirs(script_folder)
        print(f"Found {len(fonts_data['items'])} fonts. Checking top {top_popularity} for desired styles...")
        
        for font in fonts_data["items"][:top_popularity]:
            font_family = font.get("family")
            print(f"\n  > Processing font family: {font_family}")
            font_folder = os.path.join(script_folder, font_family.replace(" ", "_"))
            if not os.path.exists(font_folder): os.makedirs(font_folder)
            download_count = 0
            
            # SELECTIVE DOWNLOAD LOGIC
            for variant, url in font.get("files", {}).items():
                if variant in target_variants:
                    file_name = f"{font_family.replace(' ', '_')}-{variant}.ttf"
                    file_path = os.path.join(font_folder, file_name)
                    print(f"    > Downloading '{variant}' style...")
                    font_file_response = requests.get(url)
                    font_file_response.raise_for_status()
                    with open(file_path, "wb") as f: 
                        f.write(font_file_response.content)
                    print(f"      Saved to {file_path}")
                    download_count += 1
            
            if download_count == 0:
                print("    > None of the desired styles were found for this font family.")
    except requests.exceptions.RequestException as e:
        print(f"An error occurred for subset '{writing_system}': {e}")
    except KeyError:
        print(f"Error parsing JSON for subset '{writing_system}'.")

    print(f"\nFont download process for '{writing_system.capitalize()}' completed.")

In [33]:
indic_writing_system = [list(writing_system_to_language.keys())][0]

In [35]:
if __name__ == "__main__":
    for writing_system in indic_writing_system:
        get_specific_font_styles(writing_system, download_dir="../../fonts/")
        print(f"Completed downloading fonts for writing system: {writing_system}")
        print("-" * 50)  

Targeting styles: ['regular', 'bold', 'italic', 'bold-italic'] (API variants: ['regular', '700', 'italic', '700italic'])

--- Querying for script: Devanagari ---
Found 61 fonts. Checking top 5 for desired styles...

  > Processing font family: Noto Sans
    > Downloading 'regular' style...
      Saved to ../../fonts/Devanagari/Noto_Sans/Noto_Sans-regular.ttf
    > Downloading '700' style...
      Saved to ../../fonts/Devanagari/Noto_Sans/Noto_Sans-700.ttf
    > Downloading 'italic' style...
      Saved to ../../fonts/Devanagari/Noto_Sans/Noto_Sans-italic.ttf
    > Downloading '700italic' style...
      Saved to ../../fonts/Devanagari/Noto_Sans/Noto_Sans-700italic.ttf

  > Processing font family: Poppins
    > Downloading 'regular' style...
      Saved to ../../fonts/Devanagari/Poppins/Poppins-regular.ttf
    > Downloading 'italic' style...
      Saved to ../../fonts/Devanagari/Poppins/Poppins-italic.ttf
    > Downloading '700' style...
      Saved to ../../fonts/Devanagari/Poppins/Popp

In [34]:
indic_writing_system

['devanagari',
 'tamil',
 'telugu',
 'kannada',
 'malayalam',
 'bengali',
 'meetei-mayek',
 'gujarati',
 'gurmukhi',
 'odia',
 'arabic',
 'latin',
 'ol-chiki']